# E-Commerce Market Intelligence & A/B Testing

Two parts: (A) eBay Browse API market data, (B) A/B test statistics. Part B runs standalone with simulated data if you skip Part A entirely.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config as C
%matplotlib inline


## Part A — Collect eBay listings
Register at https://developer.ebay.com/, create an application keyset, copy `.env.example` to `.env` and paste in `EBAY_APP_ID` / `EBAY_CERT_ID` before running this cell.

In [ ]:
from src.collect import get_access_token, collect_term, flatten

token = get_access_token()
frames = []
for term in C.SEARCH_TERMS:
    items = collect_term(token, term, C.ITEMS_PER_TERM)
    frames.append(flatten(items, term))

listings = pd.concat(frames, ignore_index=True).drop_duplicates(subset="item_id")
listings.to_csv(C.LISTINGS_FILE, index=False)
print(f"{len(listings):,} listings saved")
listings.head()


## A2. Market analysis

In [ ]:
from src.market_analysis import load, price_distributions, shipping_premium, condition_gap, keyword_effect, seller_rating_effect

df = load()
price_distributions(df)
df.groupby("search_term")["price"].describe()


In [ ]:
ship = shipping_premium(df)
ship  # Welch's t-test: does free shipping carry a price premium, per category?


In [ ]:
cond = condition_gap(df)
cond  # Mann-Whitney U: new vs used price gap (prices are skewed, so a rank test)


In [ ]:
kw = keyword_effect(df)
kw  # within-category comparison, avoids Simpson's paradox


In [ ]:
seller_rating_effect(df)  # Spearman correlation: seller feedback % vs price


---
## Part B — A/B Test Analysis

Runs independently of Part A. Two modes:
- **Simulated** (default below): a known 12% true lift, useful for validating the pipeline. Always label it as simulated.
- **Real**: download the Kaggle *Marketing A/B Testing* dataset (`marketing_AB.csv`) into `data/raw/` and use `load_kaggle()` instead of `simulate()`.

In [ ]:
from src.experiment import simulate, load_kaggle, run

exp_df = simulate()          # or: exp_df = load_kaggle()
exp_df.head()


### Step 1 — Power analysis (before you'd run this for real)
How many users per arm do you need to detect a given lift?

In [ ]:
from src.ab_test import sample_size_table

baseline = exp_df[exp_df.group == "control"]["converted"].mean()
print(f"Baseline conversion rate: {baseline:.4f}")
sample_size_table(baseline)


### Step 2 — Sample ratio mismatch
If this fires, the randomisation is broken and nothing below can be trusted.

In [ ]:
from src.ab_test import srm_check

control = exp_df[exp_df.group == "control"]
treatment = exp_df[exp_df.group == "treatment"]
srm_check(len(control), len(treatment))


### Step 3 — Hypothesis tests

In [ ]:
from src.ab_test import proportion_test, continuous_test, summarise

conv_result = proportion_test(
    int(control.converted.sum()), len(control),
    int(treatment.converted.sum()), len(treatment),
)
print(summarise(conv_result))
conv_result


In [ ]:
rev_result = continuous_test(control.revenue, treatment.revenue, metric="revenue_per_user")
print(summarise(rev_result))
rev_result


### Step 4 — Multiple comparison correction
Testing several metrics at once inflates the false-positive rate. Correct for it.

In [ ]:
from src.ab_test import correct_pvalues

table = correct_pvalues([conv_result, rev_result])
table[["metric", "relative_lift_%", "p_value", "p_adjusted", "significant_after_correction"]]


### Step 5 — CUPED variance reduction
Uses a pre-experiment covariate to shrink variance, so the same effect is detectable with fewer users.

In [ ]:
from src.ab_test import cuped_adjust

adj = cuped_adjust(exp_df["revenue"].to_numpy(), exp_df["pre_spend"].to_numpy())
exp_df = exp_df.assign(revenue_cuped=adj)

raw_var, cuped_var = exp_df["revenue"].var(), exp_df["revenue_cuped"].var()
print(f"Variance {raw_var:.2f} -> {cuped_var:.2f} ({(1 - cuped_var/raw_var)*100:.1f}% reduction)")

c2 = exp_df[exp_df.group == "control"]["revenue_cuped"]
t2 = exp_df[exp_df.group == "treatment"]["revenue_cuped"]
print(summarise(continuous_test(c2, t2, metric="revenue_cuped")))


### Or just run the whole thing in one call

In [ ]:
final_table = run(exp_df, simulated=True)
final_table
